# 11a: Time Series and Decomposition

How can we examine changes in a time series without the recurring seasonal pattern obscuring them? Begin with time-index handling and moving averages using quarterly beer production, then use monthly nuclear generation to develop decomposition and seasonal adjustment. The notebook uses local data when available and loads the published copies automatically in Colab.

## References

[VanderPlas](https://jakevdp.github.io/PythonDataScienceHandbook/03.11-working-with-time-series.html) supports time-index handling. The beer data and moving-average examples draw on [FPP3 Chapters 2–3](https://otexts.com/fpp3/) and [FPP Pythonic](https://otexts.com/fpppy/data/index.html).

Adapted from Allen B. Downey, [Think Stats, third edition, §§12.1–12.2](https://allendowney.github.io/ThinkStats/chap12.html#decomposition). The local data are an extract of the EIA snapshot accompanying the book; see [data provenance](data/README.md). The reuse license appears at the end. The practical decomposition section draws on [FPP §§3.4 and 3.6](https://otexts.com/fpp3/decomposition.html#stl-decomposition) and the [statsmodels STL reference](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.STL.html).

## Setup and Data

Beer production is measured in megalitres; nuclear generation is measured in gigawatt-hours (GWh). Both series retain their seasonal variation.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose
from IPython.display import display

# Colab opens the notebook without its neighboring data folder.
data_candidates = [
    Path("data"),
    Path("lectures/11a-time-series-and-decomposition/data"),
    Path("content/lectures/11a-time-series-and-decomposition/data"),
]
data_files = ["aus-beer.csv", "us-nuclear-generation.csv"]
data_dir = next((folder for folder in data_candidates
                 if all((folder / name).exists() for name in data_files)), None)
data_url = (
    "https://raw.githubusercontent.com/olearydj/INSY7130/main/"
    "lectures/11a-time-series-and-decomposition/data/"
)


def load_data(filename):
    if data_dir is not None:
        return pd.read_csv(data_dir / filename)
    with urlopen(data_url + filename, timeout=30) as response:
        return pd.read_csv(response)

plt.rcParams.update({"font.size": 16, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
BLUE, ORANGE, GREEN = "#185c8c", "#c45116", "#16745b"

def show_plot(fig):
    for ax in fig.axes:
        if ax.get_xlabel() == "Year":
            locator = mdates.AutoDateLocator(minticks=3, maxticks=6)
            ax.xaxis.set_major_locator(locator)
            ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    plt.show()
    plt.close(fig)

## Time Series Data in Python

The slides showed patterns in beer production. To calculate with those observations, Python needs both the production values and the dates they belong to. We will read the data, make the dates usable as an index, and check that consecutive rows represent consecutive quarters.

### Observations and Their Time Index

Each row records Australian beer production during one calendar quarter, measured in megalitres. The date labels the start of that quarter: `1992-01-01` represents production over January through March, not production on January 1 alone.

Read the CSV and inspect its first five rows with `.head()`. A pandas `DataFrame` is a table with named columns; here, `ds` holds the date labels and `Beer` holds the measurements. The `.dtypes` attribute reports how pandas represents each column.

In [ ]:
frame = load_data("aus-beer.csv")
display(frame.head())
display(frame.dtypes)

The numbered index on the left identifies row positions in this imported table. We will replace it with dates. Selecting the `Beer` column will then give a `Series`: one sequence of values, each attached to a time label.

### Labels and Parsing

A date read as text is still text. Parsing converts that label into a date/time value that pandas can use for calendar selection and comparison. A pandas `Timestamp` represents one date and time; a `DatetimeIndex` holds the timestamp labels for a series.

Our file uses year-month-day labels. In `format="%Y-%m-%d"`, `%Y` specifies the four-digit year, `%m` the month, and `%d` the day. Parse that column, use it as the index, and select the production values.

See the [pandas parsing reference](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) and its linked [format-code table](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes) for other date formats.

`set_index("ds")` returns a table indexed by the parsed dates. Selecting `["Beer"]` from that table produces a Series with those dates as its index; it does not change the index of `frame`.

In [ ]:
frame["ds"] = pd.to_datetime(frame["ds"], format="%Y-%m-%d")
beer = frame.set_index("ds")["Beer"]
display(beer.head())
display(beer.index)

The index contains timestamps, but the measurements still describe whole quarters. Converting the labels does not change what was measured.

An explicit format matters when the text is ambiguous. Compare the two interpretations of these labels:

In [ ]:
labels = ["03/04/2025", "04/05/2025"]
display(pd.DataFrame({
    "Text": labels,
    "Month/day/year": pd.to_datetime(labels, format="%m/%d/%Y"),
    "Day/month/year": pd.to_datetime(labels, format="%d/%m/%Y"),
}))

`03/04/2025` can mean March 4 or April 3. The data source's convention determines which interpretation is correct.

### Frequency and Index Checks

Using a `DatetimeIndex` unlocks many powerful pandas features when working with timeseries data.

From inspection and/or the source, we know that the beer data is quarterly. To check the data quality we'll look for missing data and check the order, uniqueness, and spacing of observations.

In [ ]:
# check the data for gaps...
# first, create the expected quarterly dates from the start / end points
expected = pd.date_range(beer.index.min(), beer.index.max(), freq="QS")
# then compare that with the data to find missing observations
missing_quarters = expected.difference(beer.index)

# print diagnostics
print("Index in chronological order:", beer.index.is_monotonic_increasing)
print("Date labels unique:", beer.index.is_unique)
print("Missing expected quarters:", len(missing_quarters))
print("Missing production values:", beer.isna().sum())

# automated checks
assert beer.index.is_monotonic_increasing and beer.index.is_unique
assert beer.index.difference(expected).empty
assert missing_quarters.empty
assert beer.notna().all()

`date_range` constructs the expected dates between the first and last observations. The frequency code `QS` means quarter start; comparing that calendar with the actual index reveals missing quarters.

Read the diagnostics as follows:

- `expected.difference(beer.index)` returns expected dates absent from the observed index. Reversing the operands checks for observed dates outside the expected calendar.
- `is_monotonic_increasing` checks that dates never go backward; `is_unique` separately checks that no date is repeated.
- `isna().sum()` counts missing production values.

The `assert` statements stop execution if a required condition fails. An index's `.empty` property is true when it contains no entries; `notna().all()` is true when every production value is present.

This series is ordered, has unique dates, and has no missing quarters or production values. These checks matter because three adjacent rows represent three consecutive quarters only when the calendar is complete.

### Selecting Data in a Range of Dates

Use the date index to select a year, an exact date, or a range of months. Compare the observations returned by each selection.

In [ ]:
# all quarters in one year
display(beer.loc["1992"])

In [ ]:
# one observation by its exact date label
display(beer.loc["1992-04-01"])

In [ ]:
# observations from April through September
display(beer.loc["1992-04":"1992-09"])

`.loc` selects observations by their index labels. The year selects all four quarters; the exact date returns the single production value labeled April 1. That value describes the whole second quarter, not production on one day.

The month range includes April through the end of September, selecting Q2 and Q3. Its boundaries need not match observation dates: there is no September observation in this quarterly series (Jan, Apr, Jul, Oct).

Now select the beer observations from 1992 through 1995, the four years used in the slides.

In [ ]:
# select and display the observations
b = beer.loc["1992":"1995"]
display(b)

# check the number of observations
print("Number of observations:", len(b))
assert len(b) == 16

The slice `"1992":"1995"` includes all quarters in both endpoint years.

See the [pandas guide to date selection and partial-string indexing](https://pandas.pydata.org/docs/user_guide/timeseries.html#partial-string-indexing) for more selection examples and slicing rules.

The result contains 16 consecutive quarterly observations. We can now calculate a moving average and identify exactly which quarters contribute to each estimate.

## Moving Averages and Endpoints

Calculate a centered three-quarter moving average for the selected observations, then compare it with the original values.

In [ ]:
# calculate the centered moving average
ma3 = b.rolling(3, center=True).mean()

# compare observations and averages
display(pd.DataFrame({"Observed": b, "Centered 3-MA": ma3}).round(3))

# verify the first complete average
assert np.isclose(ma3.iloc[1], (443 + 410 + 420) / 3)

Read the calculation as follows:

- `.rolling(3)` creates overlapping windows of three observations.
- `center=True` places each result at the middle observation's date.
- `.mean()` calculates the average within each window.

The first complete average is (443 + 410 + 420) / 3 = 424.33 ML, aligned with 1992 Q2. The check uses `.iloc[1]` to select the second row by position and `np.isclose` to compare the computed value with the hand calculation, allowing for floating-point rounding.

The first and last moving-average values are `NaN` because a complete centered window is unavailable there. These are missing estimates, not missing production measurements.

The following plot compares the observed production with the moving average and marks the three observations used in the first complete calculation.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.3), layout="constrained")
ax.plot(b.index, b, "o-", color=BLUE, label="Observed")
ax.plot(b.index, ma3, "s--", color=ORANGE, label="Centered 3-MA")
ax.scatter(b.index[:3], b.iloc[:3], s=110, facecolors="none", edgecolors=GREEN, linewidths=2)
ax.annotate("First average: 424.33 ML", xy=(b.index[1], ma3.iloc[1]),
            xytext=(b.index[4], 480), arrowprops={"arrowstyle": "->"}, fontsize=14)
ax.set(ylabel="Megalitres", xlabel="Year")
ax.legend(fontsize=13)
show_plot(fig)

Averaging neighboring observations reduces the quarter-to-quarter fluctuations, producing the smoother line. Each centered estimate uses the previous, current, and next quarter. At the latest observed quarter, the next quarter is still in the future, so that centered estimate cannot yet be calculated.

## Manual Decomposition

### Monthly Electricity Generation

We now use moving averages to estimate the slower movement in a series with a recurring annual pattern. The data contain monthly US electricity generation from January 2001 through June 2024. We will examine nuclear generation, measured in gigawatt-hours (GWh). Read the local extract, parse the month labels, and plot the series.

In [ ]:
frame = load_data("us-nuclear-generation.csv")
frame["month"] = pd.to_datetime(frame["month"], format="%b %Y")
nuclear = frame.set_index("month")["generation_gwh"]
nuclear.name = "Nuclear generation"

assert nuclear.index.is_unique and nuclear.index.is_monotonic_increasing
assert nuclear.notna().all()
assert nuclear.index.equals(pd.date_range(nuclear.index.min(), nuclear.index.max(), freq="MS"))

actual_options = dict(color="C0", lw=1.3)
trend_cycle_options = dict(color="C1", lw=2)
seasonal_options = dict(color="C2", lw=1.5)
remainder_options = dict(color="C3", lw=1.3)
fitted_options = dict(color="C4", lw=2, ls="--")
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 12})

nuclear.plot(label="Nuclear generation", **actual_options)
plt.ylabel("GWh")
plt.xlabel("Year")
plt.show()
plt.close()

The month-to-month swings make slower changes harder to follow. We will estimate that slower movement first, then look for a repeating annual pattern in the variation around it.

### Decomposition Approach

We will represent each observation as the sum of a trend-cycle, a seasonal component, and a remainder. This is an additive decomposition: all three components are measured in GWh. For this example, we assume the seasonal pattern repeats unchanged each year.

1. **Estimate the slower movement.** Use a moving average to estimate the trend-cycle.
2. **Look for a repeating annual pattern.** Subtract the trend-cycle and compare the remaining values for the same month across years.
3. **Estimate the seasonal component.** Calculate a typical effect for each month and repeat those 12 effects across the series.
4. **Inspect what remains.** Subtract both estimated components from the observations. The remainder contains variation this decomposition has not explained.

The trend-cycle can rise and fall; it need not follow a straight line.

### Estimate the Trend-Cycle With a Moving Average

Average the first 12 months, January through December 2001. Then move forward one month and average February 2001 through January 2002. Continue moving the window one month at a time. Each average includes a full year of observations and smooths the within-year swings. Compare the large annual swings in the observations with the slower movement of the average.

In [ ]:
trend_cycle = nuclear.rolling(window=12).mean()

nuclear.plot(label="Observed", **actual_options)
trend_cycle.plot(label="12-month moving average", **trend_cycle_options)
plt.ylabel("GWh")
plt.legend()
plt.xlabel("Year")
plt.show()
plt.close()

This is a trailing average: the January–December 2001 average is placed at December 2001. The first 11 months have no complete window. The smoothed level rises into the late 2000s, falls around 2012–2013, and declines again after 2019. We could smooth more with a longer window, but we will keep 12 months here.

### Subtract the Trend-Cycle

Subtract the moving average from the observed generation at each date. The result is the detrended series: the amount above or below the estimated trend-cycle. `.dropna()` removes the first 11 months, where we could not calculate the average.

In [ ]:
detrended = (nuclear - trend_cycle).dropna()
departure_limit = 1.1 * detrended.abs().max()
departure_limits = (-departure_limit, departure_limit)

detrended.plot(label="Detrended generation", **actual_options)
plt.ylim(departure_limits)
plt.axhline(0, color="gray", lw=0.8)
plt.ylabel("Detrended generation (GWh)")
plt.xlabel("Year")
plt.show()
plt.close()

Positive values are above the estimated trend-cycle; negative values are below it. The repeated rises and falls remain. To describe their annual pattern, compare January with other Januaries, February with other Februaries, and so on.

### Estimate the Seasonal Pattern

Extract the calendar month from each date and group the detrended values by month. Average all the January values, then all the February values, and continue through December. This produces 12 typical departures from the trend-cycle.

In [ ]:
monthly_averages = detrended.groupby(detrended.index.month).mean()
monthly_averages.plot(marker="o", **seasonal_options)
plt.xticks(range(1, 13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                         "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
plt.xlabel("Month")
plt.ylabel("Mean detrended generation (GWh)")
plt.axhline(0, color="gray", lw=0.8)
plt.show()
plt.close()

The horizontal axis runs from January to December. The departures are higher in winter and summer and lower around April and October. Heating in winter and air conditioning in summer help explain why electricity demand has an annual pattern.

Use those 12 averages to build a seasonal series with the same dates as the observations. January 2001, January 2002, and every later January receive the January average. Each February receives the February average, and so on.

In [ ]:
# look up each date's calendar-month effect
seasonal = monthly_averages[nuclear.index.month]

# replace the repeated month-number labels with the original dates
seasonal.index = nuclear.index

`nuclear.index.month` supplies month numbers such as 1, 2, and 3. The lookup selects entries labeled with those numbers from `monthly_averages`, repeating a value whenever that month occurs again. Its index initially contains repeated month numbers; assigning `nuclear.index` attaches the corresponding dates without changing the values.

Inspect January and February in two successive years. Each row below shows a date, the month number used for lookup, and the seasonal effect assigned to it. These are the hand-built monthly effects, before the library decomposition introduced later.

In [ ]:
example_dates = pd.to_datetime([
    "2001-01-01", "2001-02-01", "2002-01-01", "2002-02-01"
])
lookup_example = pd.DataFrame({
    "Month number": example_dates.month,
    "Seasonal effect (GWh)": seasonal.loc[example_dates],
}, index=example_dates)
lookup_example.index.name = "Date"
display(lookup_example.round(2))

Both January dates receive the same effect, as do both February dates. Display 2002–2004 so that the repeating peaks and troughs are easy to follow. The seasonal series still covers the full record; only the displayed dates are restricted.

In [ ]:
display_start, display_end = "2002", "2004"
seasonal.loc[display_start:display_end].plot(label="Seasonal component", **seasonal_options)
plt.axhline(0, color="gray", lw=0.8)
plt.ylabel("Seasonal contribution (GWh)")
plt.xlabel("Year")
plt.show()
plt.close()

The plot repeats the same 12-month pattern each year, as assumed in our approach.

### Combine Trend-Cycle and Seasonality

Add the trend-cycle and seasonal component. Compare their sum with the observations over the same 2002–2004 window. Which peaks and troughs does the fitted line capture, and where do the observations depart from it? Both components were estimated using the full record; this is a closer view of the result.

In [ ]:
fitted = trend_cycle + seasonal

nuclear.loc[display_start:display_end].plot(label="Observed", **actual_options)
fitted.loc[display_start:display_end].plot(label="Trend-cycle + seasonal", **fitted_options)
plt.ylabel("GWh")
plt.legend()
plt.xlabel("Year")
plt.show()
plt.close()

The fitted line follows the January and summer peaks and the spring and autumn troughs. Their heights and depths do not match every observation. These are fitted historical values, not forecasts.

### Inspect the Remainder

Subtract the fitted line from the observations to calculate the remainder: the amount above or below the trend-cycle plus seasonal effect.

In [ ]:
remainder = nuclear - fitted
remainder.plot(label="Remainder", **remainder_options)
plt.ylim(departure_limits)
plt.axhline(0, color="gray", lw=0.8)
plt.ylabel("Unexplained departure (GWh)")
plt.xlabel("Year")
plt.show()
plt.close()

The strong annual oscillation in the detrended plot is much reduced. These two plots use the same vertical limits, so the smaller departures can be compared directly. Large departures remain, including the negative spike around 2012.

We have separated the slower movement and recurring annual pattern asked about at the start. What remains can include unusually hot or cold weather, generators offline for maintenance, and changes in demand. These are possible explanations to investigate, not causes identified by the calculation. The remainder can also contain patterns our estimates missed.

## Use a Library Decomposition

Statsmodels provides `seasonal_decompose` to estimate these three components. Apply it to the same nuclear-generation series.

In [ ]:
decomposition = seasonal_decompose(nuclear, model="additive", period=12)

fig = decomposition.plot()
for ax, label, color in zip(
    fig.axes,
    ["Observed", "Trend-cycle", "Seasonal", "Remainder"],
    ["C0", "C1", "C2", "C3"],
):
    ax.set_ylabel(label)
    for line in ax.lines:
        line.set_color(color)
    if label in ["Seasonal", "Remainder"]:
        ax.axhline(0, color="gray", lw=0.8)
fig.axes[0].set_title("")
fig.axes[-1].set_xlabel("Year")
fig.set_size_inches(10, 8)
fig.suptitle("Nuclear generation: additive decomposition (GWh)")
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()
plt.close(fig)

`model="additive"` means the components combine by addition. `period=12` specifies an annual cycle of 12 monthly observations. The returned object holds the components separately: `.trend` contains the trend-cycle, `.seasonal` the seasonal component, and `.resid` the remainder.

Read the panels in that order: slower movement, repeating annual effects, then departures left over. The panels have different vertical scales, but all are measured in GWh.

### Observed Generation and Fitted Components

The four-panel plot shows the raw decomposition components. The overlay below shows how the trend-cycle and seasonal component combine to approximate observed generation on the original GWh scale. Show 2007–2016 to follow the decline around 2012 and subsequent recovery while keeping individual seasonal cycles visible. The gray line shows observed generation, the orange line shows the trend-cycle, and the blue line adds the seasonal component to the trend-cycle.

In [ ]:
library_fitted = decomposition.trend + decomposition.seasonal
overlay_start, overlay_end = "2007", "2016"

fig, ax = plt.subplots(figsize=(12, 5))
nuclear.loc[overlay_start:overlay_end].plot(
    ax=ax, color="0.65", lw=1.3, label="Observed"
)
decomposition.trend.loc[overlay_start:overlay_end].plot(
    ax=ax, color="C1", lw=2.5, label="Trend-cycle"
)
library_fitted.loc[overlay_start:overlay_end].plot(
    ax=ax, color="C0", lw=1.8, label="Trend-cycle + seasonal"
)
ax.set(
    xlabel="Year", ylabel="Generation (GWh)"
)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
fig.tight_layout()
plt.show()
plt.close(fig)

At each date, the vertical difference between the blue and orange lines is the seasonal effect. The difference between the gray and blue lines is the remainder: observed generation minus the fitted components. The components were estimated using the full record; this overlay displays ten years of those estimates.

The broad pattern is similar, but the components differ from our calculations in two ways:

- The library centers its trend-cycle average, using observations before and after a date. Our average used that month and the preceding 11 months.
- It adjusts the 12 seasonal effects to average zero, keeping the overall level in the trend-cycle.

The endpoint gaps make the centering difference visible. Our trailing estimate begins in December 2001, losing the first 11 months. The library trend-cycle begins in July 2001 and ends in December 2023: six months are missing at each end because the centered calculation needs observations on both sides. The library remainder has the same gaps.

These estimation choices explain why the library does not reproduce our component values exactly.

### Reconstruct One Observation

Inspect January 2002, a month with all three library-estimated components available. The row below puts the observed generation beside its components and their sum, all in GWh. The `min_count=3` setting requires all three components before reporting a reconstructed value.

In [ ]:
components = pd.DataFrame({
    "Observed": nuclear,
    "Trend-cycle": decomposition.trend,
    "Seasonal": decomposition.seasonal,
    "Remainder": decomposition.resid,
})
components["Reconstructed"] = components[["Trend-cycle", "Seasonal", "Remainder"]].sum(
    axis=1, min_count=3
)
display(components.loc[["2002-01-01"]].round(2))

complete = components.dropna()
np.testing.assert_allclose(complete["Observed"], complete["Reconstructed"])

For January 2002, the estimated trend-cycle is 64,605.04 GWh. The January seasonal effect adds 6,340.26 GWh, giving a fitted value of 70,945.30 GWh. Observed generation was 70,926.00 GWh, so the remainder is −19.30 GWh: production was slightly below the fitted value.

Using the displayed values:

$$
64{,}605.04 + 6{,}340.26 - 19.30 = 70{,}926.00
$$

The seasonal effect accounts for most of this month's elevation above the trend-cycle. The remainder accounts for the remaining difference. Reconstruction is exact because the remainder was defined by subtraction; it does not establish forecast accuracy.

## Classical Limitations and STL

The manual calculation made the decomposition steps visible, and classical decomposition packaged them into a library call. For practical decomposition in this course, we will use STL. It keeps the same component meanings while giving us more control over how they are estimated.

### Limits of the Earlier Approach

Three limitations follow from the calculations we used:

- **The seasonal pattern is fixed.** Every January receives the same effect, whether it is 2001 or 2024. Changes in that pattern must go into the trend-cycle or remainder instead.
- **Moving averages need complete windows.** Our trailing average lost the first 11 months and lagged changes in level. The centered classical estimate lost six months at each end. A fixed window can also smooth over a rapid change.
- **Unusual observations influence the averages.** A very low production month can pull down both the trend estimate and the seasonal average for its calendar month. Some of that unusual departure can be absorbed into the components we are trying to estimate.

These are reasons to use a more flexible estimator, rather than repeat the manual procedure for every new series.

### STL: Local Smoothing and Robust Fitting

STL means *Seasonal and Trend decomposition using Loess*. Loess estimates a smooth curve by fitting simple relationships to nearby observations, giving more influence to closer observations. STL alternates between estimating seasonal and trend-cycle structure.

This changes what the decomposition can do:

- Seasonal effects can evolve. A January estimate borrows information from nearby Januaries rather than assigning one average to every year.
- Smoothing settings control how quickly the components can change. Local fits also provide endpoint estimates using the observations available there, although endpoints have less balanced support than interior dates.
- Robust fitting reduces the influence of observations with large remainders when estimating the components. Their departures remain visible in the remainder instead of pulling the fitted curves as strongly.

We will use an additive STL decomposition on the original GWh scale, with robust fitting enabled.

### Fit the Nuclear-Generation Series

Create an STL specification for the same monthly series, then call `.fit()` to estimate its components.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl_model = STL(nuclear, period=12, seasonal=13, robust=True)
stl = stl_model.fit()

# check that the components cover the record and reconstruct the observations
assert stl.trend.notna().all()
assert stl.seasonal.notna().all()
assert stl.resid.notna().all()
np.testing.assert_allclose(stl.trend + stl.seasonal + stl.resid, nuclear)

The settings have different jobs:

- `period=12` specifies the annual cycle: 12 monthly observations.
- `seasonal=13` controls smoothing across successive occurrences of the same calendar month. For monthly data, it uses a neighborhood of 13 annual occurrences, such as nearby Januaries. This deliberately favors gradual seasonal change over year-to-year fluctuations; it is an illustrative choice, not an optimized setting. The window must be odd. The statsmodels default is 7.
- `robust=True` enables the reweighting of unusual observations. The default is `False`.
- We leave the trend smoother at its library-selected setting. With these choices, statsmodels uses a 21-observation trend window. Unlike `seasonal`, that window refers to consecutive months.

The fitted result exposes `.trend`, `.seasonal`, and `.resid`, just as the classical result did. The meanings and units of those components have not changed.

### Compare Classical Decomposition and STL

Compare the two methods over 2007–2016, using the same date range as the earlier overlay. Each panel puts both methods on the same scale. Look for differences in the slower level, changes in the seasonal pattern, and observations left in the remainder.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, field, label, color in zip(
    axes,
    ["trend", "seasonal", "resid"],
    ["Trend-cycle", "Seasonal", "Remainder"],
    ["C1", "C2", "C3"],
):
    classical_values = getattr(decomposition, field).loc[overlay_start:overlay_end]
    stl_values = getattr(stl, field).loc[overlay_start:overlay_end]
    classical_values.plot(ax=ax, color="0.45", ls="--", lw=1.3, label="Classical")
    stl_values.plot(ax=ax, color=color, lw=1.5, label="STL (robust)")
    ax.set_ylabel(f"{label} (GWh)")
    if field != "trend":
        ax.axhline(0, color="gray", lw=0.7)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Year")
fig.suptitle("Nuclear generation: classical and STL components")
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()
plt.close(fig)

Use STL as the practical starting point, then inspect whether its components make sense for the series. Its flexibility does not make the estimated components uniquely true, and a small remainder alone is not the goal.

## Seasonal Adjustment

Is nuclear generation declining or recovering, and which months depart from that broader movement? The recurring annual peaks and troughs make those changes harder to see. We want to examine generation with the usual seasonal pattern removed.

Seasonal adjustment does this by subtracting the estimated seasonal component from each observation. We use the seasonal estimates from our STL fit:

$$
\text{Seasonally adjusted} = \text{Observed} - \text{Seasonal} = \text{Trend-cycle} + \text{Remainder}.
$$

In [ ]:
adjusted = nuclear - stl.seasonal

The adjusted series is still measured in GWh. For a month with a positive seasonal effect, we subtract that expected boost; for a month with a negative seasonal effect, subtracting it raises the adjusted value. This puts different months on a more comparable basis.

Plot the adjusted series alongside the observations and the trend-cycle over the same 2007–2016 window. Look for the broader changes and the departures that remain after removing seasonality.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
nuclear.loc[overlay_start:overlay_end].plot(
    ax=ax, color="0.7", lw=1.3, label="Observed"
)
adjusted.loc[overlay_start:overlay_end].plot(
    ax=ax, color="C0", lw=1.5, label="Seasonally adjusted"
)
stl.trend.loc[overlay_start:overlay_end].plot(
    ax=ax, label="Trend-cycle", **trend_cycle_options
)
ax.set_ylabel("Generation (GWh)")
ax.set_xlabel("Year")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
fig.tight_layout()
plt.show()
plt.close(fig)

The adjusted series shows the decline into 2012 and the recovery afterward without the large recurring annual swings. It also retains unusual departures, including the sharp drop in May 2011. That drop is still present because the estimated seasonal pattern does not account for it.

Seasonally adjusted generation still fluctuates around the trend-cycle: those departures are the remainder. Keeping them lets us examine both broader changes and unusual months. Removing seasonality helps us see those changes; explaining their causes requires other evidence.

## Connection to Dependence

After removing the estimated trend-cycle and seasonality, are observations in the remainder independent, or do they still have time dependencies? For example, does a positive remainder tend to be followed by another positive remainder? If so, the decomposition has left some time-related structure unexplained.

In the next lecture, we will use lags and autocorrelation to investigate that remaining dependence. An absence of autocorrelation alone does not establish independence.

## Reuse Notice

MIT License

Copyright (c) 2024 Allen Downey

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.